# Split objects into broad lineages

Writes one raw-count object per compartment (myeloid, lymphoid, malignant epithelial/alveolar)
from the final annotated atlas, as input to the compartment-level notebooks.

In [ ]:
# !pip install --quiet scanpy
# %cd <project>/notebooks

In [ ]:
import os

import numpy as np
import scanpy as sc

## Config

In [ ]:
input_path = "./../snRNAseq/output/cellbender/cleanup/scVI_integrated_adata_clean_latent20_major_cell_type_inferCNV_add_copyKAT_score_ctype_new.h5ad"

output_dir = "./../snRNAseq/output/cellbender/cleanup"
label_col = "cell_type_new"
counts_layer = "raw_counts"

# output file name -> labels included
compartments = {
    "sub_Myeloid_raw.h5ad": ["Myeloid"],
    "sub_T_NK_cells_raw.h5ad": ["Lymphoid"],              # also contains B and plasma cells
    "sub_Epi_Tumor_Alveolar_raw.h5ad": ["Malignant", "Non-malignant epithelial", "Alveolar cell type I",
                                        "Alveolar cell type II", "Transitional", "Goblet cell", "Club cell"],
}

## 1. Load (X = raw counts)

In [ ]:
adata = sc.read(input_path)
adata.X = adata.layers[counts_layer].copy()
vals = adata.X[:1000].data
assert np.allclose(vals, np.round(vals)), "X must contain raw counts"
adata.obs[label_col].value_counts()

## 2. Write one object per compartment

In [ ]:
labels = set(adata.obs[label_col])
for fname, keep in compartments.items():
    missing = set(keep) - labels
    assert not missing, f"{fname}: labels not found: {sorted(missing)}"
    sub = adata[adata.obs[label_col].isin(keep)].copy()
    sub.obs[label_col] = sub.obs[label_col].cat.remove_unused_categories()
    sub.write(os.path.join(output_dir, fname))
    print(f"{fname}: {sub.n_obs} cells | {sub.obs[label_col].value_counts().to_dict()}")

used = {l for keep in compartments.values() for l in keep}
print("labels not written to any compartment:", sorted(labels - used))